# SC H.4216 Tax Reform Analysis - Test Dataset

This notebook produces analysis in the same format as the RFA fiscal note for direct comparison.

**Dataset:** `hf://policyengine/test/mar/SC.h5`

**Reform:** H.4216 with 5.39% top rate (RFA version)

In [1]:
from policyengine_us import Microsimulation
from policyengine_us.reforms.states.sc.h4216.sc_h4216 import create_sc_h4216
from policyengine_core.reforms import Reform
import pandas as pd
import numpy as np

SC_DATASET = "hf://policyengine/test/mar/SC.h5"
TAX_YEAR = 2026

In [2]:
def create_h4216_reform(top_rate=0.0539):
    """
    SC H.4216 Reform:
    - 1.99% up to $30k
    - top_rate over $30k (default 5.39% for RFA comparison)
    """
    param_reform = Reform.from_dict(
        {
            "gov.contrib.states.sc.h4216.in_effect": {
                "2026-01-01.2100-12-31": True
            },
            "gov.contrib.states.sc.h4216.rates[1].rate": {
                "2026-01-01.2100-12-31": top_rate
            }
        },
        country_id="us",
    )
    base_reform = create_sc_h4216()
    return (base_reform, param_reform)

print("Loading simulations...")
baseline = Microsimulation(dataset=SC_DATASET)
reform_sim = Microsimulation(dataset=SC_DATASET, reform=create_h4216_reform(0.0539))
print("Done!")

Loading simulations...
Done!


In [3]:
# Get data - use .values to avoid double-weighting
baseline_tax = baseline.calculate("sc_income_tax", period=TAX_YEAR, map_to="tax_unit").values
reform_tax = reform_sim.calculate("sc_income_tax", period=TAX_YEAR, map_to="tax_unit").values
agi = baseline.calculate("adjusted_gross_income", period=TAX_YEAR, map_to="tax_unit").values
weight = baseline.calculate("tax_unit_weight", period=TAX_YEAR).values

tax_change = reform_tax - baseline_tax

print(f"Total tax units: {len(baseline_tax):,}")
print(f"Weighted tax units: {weight.sum():,.0f}")

Total tax units: 42,461
Weighted tax units: 2,705,850


In [4]:
# Define income brackets matching RFA exactly
income_brackets = [
    (float('-inf'), 0, "$0*"),
    (0, 10000, "$1 to $10000"),
    (10000, 20000, "$10001 to $20000"),
    (20000, 30000, "$20001 to $30000"),
    (30000, 40000, "$30001 to $40000"),
    (40000, 50000, "$40001 to $50000"),
    (50000, 75000, "$50001 to $75000"),
    (75000, 100000, "$75001 to $100000"),
    (100000, 150000, "$100001 to $150000"),
    (150000, 200000, "$150001 to $200000"),
    (200000, 300000, "$200001 to $300000"),
    (300000, 500000, "$300001 to $500000"),
    (500000, 1000000, "$500001 to $1000000"),
    (1000000, float('inf'), "Over $1000000")
]

total_weight = weight.sum()
results = []

for lower, upper, label in income_brackets:
    if lower == float('-inf'):
        mask = agi <= upper
    elif upper == float('inf'):
        mask = agi > lower
    else:
        mask = (agi > lower) & (agi <= upper)
    
    if mask.sum() == 0:
        continue
    
    # Basic stats
    est_returns = weight[mask].sum()
    pct_returns = est_returns / total_weight * 100
    
    old_avg_tax = np.average(baseline_tax[mask], weights=weight[mask]) if est_returns > 0 else 0
    new_avg_tax = np.average(reform_tax[mask], weights=weight[mask]) if est_returns > 0 else 0
    
    # Returns with tax change (threshold $1)
    change_mask = mask & (np.abs(tax_change) > 1)
    returns_with_change = weight[change_mask].sum()
    pct_with_change = returns_with_change / est_returns * 100 if est_returns > 0 else 0
    
    if returns_with_change > 0:
        old_avg_changed = np.average(baseline_tax[change_mask], weights=weight[change_mask])
        new_avg_changed = np.average(reform_tax[change_mask], weights=weight[change_mask])
        avg_change = np.average(tax_change[change_mask], weights=weight[change_mask])
    else:
        old_avg_changed = 0
        new_avg_changed = 0
        avg_change = 0
    
    total_change = (tax_change[mask] * weight[mask]).sum()
    
    # Tax decrease
    decrease_mask = mask & (tax_change < -1)
    decrease_returns = weight[decrease_mask].sum()
    decrease_pct = decrease_returns / est_returns * 100 if est_returns > 0 else 0
    total_decrease = (tax_change[decrease_mask] * weight[decrease_mask]).sum() if decrease_returns > 0 else 0
    avg_decrease = np.average(tax_change[decrease_mask], weights=weight[decrease_mask]) if decrease_returns > 0 else 0
    
    # Tax increase
    increase_mask = mask & (tax_change > 1)
    increase_returns = weight[increase_mask].sum()
    increase_pct = increase_returns / est_returns * 100 if est_returns > 0 else 0
    total_increase = (tax_change[increase_mask] * weight[increase_mask]).sum() if increase_returns > 0 else 0
    avg_increase = np.average(tax_change[increase_mask], weights=weight[increase_mask]) if increase_returns > 0 else 0
    
    # No change
    no_change_mask = mask & (np.abs(tax_change) <= 1)
    no_change_returns = weight[no_change_mask].sum()
    no_change_pct = no_change_returns / est_returns * 100 if est_returns > 0 else 0
    
    # Zero tax
    zero_tax_mask = mask & (reform_tax <= 0)
    zero_tax_returns = weight[zero_tax_mask].sum()
    zero_tax_pct = zero_tax_returns / est_returns * 100 if est_returns > 0 else 0
    
    results.append({
        "Federal AGI Range": label,
        "Est # Returns": int(round(est_returns)),
        "Est % Returns": f"{pct_returns:.1f}%",
        "Old Avg Tax Liability": f"${int(round(old_avg_tax))}",
        "New Avg Tax Liability": f"${int(round(new_avg_tax))}",
        "Returns with Tax Change": int(round(returns_with_change)),
        "% Returns in Range with Change": f"{pct_with_change:.1f}%",
        "Old Avg Tax (Changed)": f"${int(round(old_avg_changed))}",
        "New Avg Tax (Changed)": f"${int(round(new_avg_changed))}",
        "Avg Tax Change": f"${int(round(avg_change))}",
        "Total Dollar Change": f"${int(round(total_change))}",
        "Tax Decrease # Returns": int(round(decrease_returns)),
        "Tax Decrease % in Range": f"{decrease_pct:.1f}%",
        "Total Decrease Amount": f"${int(round(total_decrease))}",
        "Avg Decrease Amount": f"${int(round(avg_decrease))}",
        "Tax Increase # Returns": int(round(increase_returns)),
        "Tax Increase % in Range": f"{increase_pct:.1f}%",
        "Total Increase Amount": f"${int(round(total_increase))}",
        "Avg Increase Amount": f"${int(round(avg_increase))}",
        "No Tax Change # Returns": int(round(no_change_returns)),
        "No Change % Returns": f"{no_change_pct:.1f}%",
        "Zero Tax # Returns": int(round(zero_tax_returns)),
        "Zero Tax % Returns": f"{zero_tax_pct:.1f}%"
    })

print("Bracket analysis complete!")

Bracket analysis complete!


In [5]:
# Calculate totals
change_mask_all = np.abs(tax_change) > 1
decrease_mask_all = tax_change < -1
increase_mask_all = tax_change > 1
no_change_mask_all = np.abs(tax_change) <= 1
zero_tax_mask_all = reform_tax <= 0

total_old_avg = np.average(baseline_tax, weights=weight)
total_new_avg = np.average(reform_tax, weights=weight)
total_change_amount = (tax_change * weight).sum()

returns_with_change_all = weight[change_mask_all].sum()
old_avg_changed_all = np.average(baseline_tax[change_mask_all], weights=weight[change_mask_all]) if returns_with_change_all > 0 else 0
new_avg_changed_all = np.average(reform_tax[change_mask_all], weights=weight[change_mask_all]) if returns_with_change_all > 0 else 0
avg_change_all = np.average(tax_change[change_mask_all], weights=weight[change_mask_all]) if returns_with_change_all > 0 else 0

decrease_returns_all = weight[decrease_mask_all].sum()
total_decrease_all = (tax_change[decrease_mask_all] * weight[decrease_mask_all]).sum()
avg_decrease_all = np.average(tax_change[decrease_mask_all], weights=weight[decrease_mask_all]) if decrease_returns_all > 0 else 0

increase_returns_all = weight[increase_mask_all].sum()
total_increase_all = (tax_change[increase_mask_all] * weight[increase_mask_all]).sum()
avg_increase_all = np.average(tax_change[increase_mask_all], weights=weight[increase_mask_all]) if increase_returns_all > 0 else 0

no_change_returns_all = weight[no_change_mask_all].sum()
zero_tax_returns_all = weight[zero_tax_mask_all].sum()

results.append({
    "Federal AGI Range": "Total",
    "Est # Returns": int(round(total_weight)),
    "Est % Returns": "100.0%",
    "Old Avg Tax Liability": f"${int(round(total_old_avg))}",
    "New Avg Tax Liability": f"${int(round(total_new_avg))}",
    "Returns with Tax Change": int(round(returns_with_change_all)),
    "% Returns in Range with Change": f"{returns_with_change_all / total_weight * 100:.1f}%",
    "Old Avg Tax (Changed)": f"${int(round(old_avg_changed_all))}",
    "New Avg Tax (Changed)": f"${int(round(new_avg_changed_all))}",
    "Avg Tax Change": f"${int(round(avg_change_all))}",
    "Total Dollar Change": f"${int(round(total_change_amount))}",
    "Tax Decrease # Returns": int(round(decrease_returns_all)),
    "Tax Decrease % in Range": f"{decrease_returns_all / total_weight * 100:.1f}%",
    "Total Decrease Amount": f"${int(round(total_decrease_all))}",
    "Avg Decrease Amount": f"${int(round(avg_decrease_all))}",
    "Tax Increase # Returns": int(round(increase_returns_all)),
    "Tax Increase % in Range": f"{increase_returns_all / total_weight * 100:.1f}%",
    "Total Increase Amount": f"${int(round(total_increase_all))}",
    "Avg Increase Amount": f"${int(round(avg_increase_all))}",
    "No Tax Change # Returns": int(round(no_change_returns_all)),
    "No Change % Returns": f"{no_change_returns_all / total_weight * 100:.1f}%",
    "Zero Tax # Returns": int(round(zero_tax_returns_all)),
    "Zero Tax % Returns": f"{zero_tax_returns_all / total_weight * 100:.1f}%"
})

df_results = pd.DataFrame(results)
print("Totals calculated!")

Totals calculated!


In [6]:
# Display summary
print("="*100)
print("H.4216 - POLICYENGINE ANALYSIS (Test Dataset, 5.39% Top Rate)")
print("="*100)
print(f"\nTotal Returns: {int(total_weight):,}")
print(f"General Fund Impact: ${total_change_amount:,.0f}")
print(f"\nRFA Estimate: -$119,100,000")
print(f"Difference: ${total_change_amount - (-119100000):,.0f}")
print(f"Accuracy: {(1 - abs(total_change_amount - (-119100000)) / 119100000) * 100:.1f}%")
print("="*100)

H.4216 - POLICYENGINE ANALYSIS (Test Dataset, 5.39% Top Rate)

Total Returns: 2,705,849
General Fund Impact: $-92,710,912

RFA Estimate: -$119,100,000
Difference: $26,389,088
Accuracy: 77.8%


In [7]:
# Export to CSV in RFA format
df_results.to_csv('pe_h4216_test_analysis.csv', index=False)
print("Exported to: pe_h4216_test_analysis.csv")

Exported to: pe_h4216_test_analysis.csv


In [8]:
# Display key columns for quick comparison
display_cols = [
    "Federal AGI Range", "Est # Returns", "Est % Returns",
    "Old Avg Tax Liability", "New Avg Tax Liability", "Total Dollar Change"
]
print("\nKEY METRICS:")
print(df_results[display_cols].to_string(index=False))


KEY METRICS:
  Federal AGI Range  Est # Returns Est % Returns Old Avg Tax Liability New Avg Tax Liability Total Dollar Change
                $0*         727881         26.9%                    $0                    $0                  $0
       $1 to $10000         498186         18.4%                    $0                    $0                  $0
   $10001 to $20000         233000          8.6%                    $0                    $4             $847688
   $20001 to $30000         171515          6.3%                   $40                   $56            $2756262
   $30001 to $40000         157010          5.8%                  $149                  $135           $-2140517
   $40001 to $50000         132402          4.9%                  $399                  $302          $-12807614
   $50001 to $75000         245406          9.1%                  $701                  $584          $-28577564
  $75001 to $100000         165713          6.1%                 $1452            

## Side-by-Side Comparison with RFA

In [9]:
# Load RFA data
rfa_df = pd.read_csv('../rfa_h4216_analysis.csv')

def parse_dollar(val):
    if isinstance(val, str):
        return float(val.replace('$', '').replace(',', '').replace('%', ''))
    return val

def parse_pct(val):
    if isinstance(val, str):
        return float(val.replace('%', ''))
    return val

# Create comparison
comparison = []
for idx, pe_row in df_results.iterrows():
    agi_range = pe_row['Federal AGI Range']
    rfa_match = rfa_df[rfa_df['Federal AGI Range'] == agi_range]
    
    pe_returns = pe_row['Est # Returns']
    pe_impact = parse_dollar(pe_row['Total Dollar Change'])
    
    if len(rfa_match) > 0:
        rfa_returns = rfa_match['Est # Returns'].values[0]
        rfa_impact = parse_dollar(rfa_match['Total Dollar Change'].values[0])
    else:
        rfa_returns = 0
        rfa_impact = 0
    
    comparison.append({
        'AGI Range': agi_range,
        'PE Returns': f"{pe_returns:,}",
        'RFA Returns': f"{rfa_returns:,}" if rfa_returns else "N/A",
        'PE Impact': f"${pe_impact:,.0f}",
        'RFA Impact': f"${rfa_impact:,.0f}" if rfa_impact else "N/A",
        'Diff': f"${pe_impact - rfa_impact:+,.0f}" if rfa_impact else "N/A"
    })

comparison_df = pd.DataFrame(comparison)
print("\n" + "="*100)
print("POLICYENGINE (Test) vs RFA COMPARISON (5.39% Rate)")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)


POLICYENGINE (Test) vs RFA COMPARISON (5.39% Rate)
          AGI Range PE Returns RFA Returns     PE Impact    RFA Impact         Diff
                $0*    727,881      78,854            $0     $-571,000    $+571,000
       $1 to $10000    498,186     286,253            $0    $1,655,000  $-1,655,000
   $10001 to $20000    233,000     310,122      $847,688    $2,872,000  $-2,024,312
   $20001 to $30000    171,515     275,560    $2,756,262      $769,000  $+1,987,262
   $30001 to $40000    157,010     269,566   $-2,140,517  $-19,360,000 $+17,219,483
   $40001 to $50000    132,402     234,386  $-12,807,614  $-41,986,000 $+29,178,386
   $50001 to $75000    245,406     407,593  $-28,577,564  $-82,146,000 $+53,568,436
  $75001 to $100000    165,713     250,437  $-26,753,744  $-36,461,000  $+9,707,256
 $100001 to $150000    225,396     298,343   $49,609,656    $3,115,000 $+46,494,656
 $150001 to $200000     42,792     143,398   $32,593,342   $50,933,000 $-18,339,658
 $200001 to $300000     

In [10]:
# Full results table
print("\n" + "="*120)
print("FULL POLICYENGINE ANALYSIS (RFA Format)")
print("="*120)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(df_results.to_string(index=False))


FULL POLICYENGINE ANALYSIS (RFA Format)
  Federal AGI Range  Est # Returns Est % Returns Old Avg Tax Liability New Avg Tax Liability  Returns with Tax Change % Returns in Range with Change Old Avg Tax (Changed) New Avg Tax (Changed) Avg Tax Change Total Dollar Change  Tax Decrease # Returns Tax Decrease % in Range Total Decrease Amount Avg Decrease Amount  Tax Increase # Returns Tax Increase % in Range Total Increase Amount Avg Increase Amount  No Tax Change # Returns No Change % Returns  Zero Tax # Returns Zero Tax % Returns
                $0*         727881         26.9%                    $0                    $0                        0                           0.0%                    $0                    $0             $0                  $0                       0                    0.0%                    $0                  $0                       0                    0.0%                    $0                  $0                   727881              100.0%              7